## Notebook map

| # | Notebook | Content |
|---|---|---|
| 01 | `explorar_lndb.ipynb` | LNDb: exploration and the LNDb nodule table |
| 02 | `lidc_exploration.ipynb` | LIDC-IDRI: exploration and the LIDC nodule table |
| 03 | `03_tabela_nodulos.ipynb` | Joins both tables, label views |
| 04 | `04_patches.ipynb` | 3D patches and their checks |
| 05 | `05_features.ipynb` | Segmentation, handcrafted features, PyRadiomics, ICC |
| 06 | `06_modelo.ipynb` | Ordinal model, size model, ComBat, validation, submission |

Shared code: `src/mass_detect/note/` (`config`, `schema`, `lidc`, `patches`, `qc`, `features`, `radiomics_features`, `models`, `validation`). Unit tests: `pytest src/mass_detect/tests`.

<div style="font-family: Arial, sans-serif; line-height: 1.6; color: #fab956; max-width: 1200px; margin: 0 auto; padding: 20px; border: 1px solid #000000; border-radius: 8px; background-color: #00183a74;">

  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    Predicting Lung Nodule Malignancy from CT Data
  </h1>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   What data do we have?
  </h3>

  <p>
  
| Dataset | What it contains | How we use it |
|---|---|---|
| LNDb train | 236 CT scans, with 1219 unique findings (768 nodules), segmentation masks for nodules >= 3 mm | Main training data |
| LNDb test | 58 CT scans and 2900 candidates (50 per CT) with only a centroid | Inference only. Every candidate is predicted as if it were a nodule |
| LIDC IDRI | 1010 patients, 1018 CT series, up to four readings per nodule | Annotations through `pylidc`; patches cut from the DICOM files; used to train the size model |
  </p>
  <p>
  Malignancy is evaluated in LNDb's file <strong>'chars_trainNodules.csv'</strong>.
  </p>
  
  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
    Which files to consider? And which to discard?
  </h3>

  <ol style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">Kept:
      <ul style="padding-left: 20px; margin-bottom: 20px;">
      <li style="margin-bottom: 8px;">The merged findings file <strong>'trainNodules_gt.csv'</strong>. The unit of analysis is the nodule, not the reading, because one nodule can be read by up to three radiologists and counting it three times would leak information which is a crucial error we made sure not to make.</li>
      <li style="margin-bottom: 8px;"><strong>'chars_trainNodules.csv'</strong> for the malignancy ratings, <strong>'LNDbAcqParams.csv'</strong> for scanner information, and the masks for nodules of 3 mm or more.</li>
      <li style="margin-bottom: 8px;">The official LNDb folds <strong>'trainFolds.csv'</strong>, split by CT, used to confirm the final choice on splits that played no part in it.</li>
      </ul>
    </li>
    <li style="margin-bottom: 8px;">Dropped:
      <ul style="padding-left: 20px; margin-bottom: 20px;">
        <li style="margin-bottom: 8px;">The per patient diagnosis file of LIDC. It says whether a patient has cancer but not which nodule is the malignant one, so using it as a label would give every nodule of that patient the same wrong answer. Therefore, we did not consider it useful for our main goal.</li>
        <li style="margin-bottom: 8px;">Fleischner files and medical reports. Fleischner is a follow up recommendation, not malignancy, and reports do not exist for the test scans.</li>
        <li style="margin-bottom: 8px;">The original <strong>'.rar'</strong> archives, the X ray series of LIDC (DX and CR) and the macOS <strong>'._*'</strong> files.</li>
        </ul>
    </li>
    <li style="margin-bottom: 8px;">Practical Notes:</li>
      <ul style="padding-left: 20px; margin-bottom: 20px;">
        <li style="margin-bottom: 8px;">In the LNDb files, Malignancy is averaged in <strong>'allNods.csv'</strong>.</li>
      </ul>
    </li>
  </ol>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Key Findings in the Literature
  </h3>

  <p>
  <strong> Size alone explains a lot -</strong> Baltatzis et al. (2021) show that a baseline that only uses nodule diameter reaches 83% to 85% accuracy on LIDC, and that the way labels are aggregated can matter more than the model. Mean and median aggregation give 1010 and 848 usable nodules respectively. We will always report a diameter only baseline and use that for any comparision needed.
  </p>
  <p>
  <strong>Splits must respect patients -</strong> Causey et al. criticise works that mix nodules of the same patient across training and validation. Nodules from one scan share scanner, protocol and anatomy, so we we decided to always validate by CT scan.
  </p>
  <p>
  <strong>Tools -</strong> PyRadiomics (van Griethuysen et al., 2017) is our feature extractor and the IBSI work (Zwanenburg et al., 2020) is the reference for feature definitions. We use a fixed bin width and resample to 1 mm isotropic voxels.
  </p>
  <p>
  <strong>Domain shift -</strong> LNDb comes from one hospital with three Siemens scanner models, while LIDC is multi centre and multi vendor. LNDb was designed as an external set for models trained on LIDC. We first rely on resampling and normalisation, and only consider ComBat harmonisation (Orlhac et al., 2022) if the features show a clear scanner effect.
  </p>
  <p>
  <strong>Pretrained features -</strong> The foundation model of Pai et al. (2024) only needs the lesion centroid, exactly what our test file has. It is an option for later, since our team decided not to use deep learning for now.
  </p>

  <h3 style="color: #f0f0f0; font-weight: normal; margin-top: -5px; margin-bottom: 20px;">
   Decisions
  </h3>

  | Decision | Choice | Why |
|----|----|----|
| Unit of analysis | Unique nodule | Avoids counting the same nodule several times |
| Training rows | Nodules with malignancy rating only | Non-nodules have no rating and only add noise |
| Training label | Mean of the non zero ratings, rounded half up | Keeps the 1 to 5 scale |
| Region of interest | Same procedure from the centroid in training and test | The test set has only coordinates. Masks serve only to check our segmentation and robustness of features |
| Features | 21 handcrafted features and 9 sphere features; PyRadiomics analysed (IBSI configuration, ICC) but not used because it did not help | Simple, explainable, robust |
| Size | Learned from the radiologists' diameters (exist only in training) | Segmentation measures size poorly; learning using privileged information (Vapnik and Vashist, 2009) |
| Model | Ordinal logistic regression with an L2 penalty | Ordered classes, few parameters (McCullagh, 1980) |
| Decision rule | Most probable class after a partial correction for class frequencies (power 0.5 for the rarest classes) | Balances the four competition metrics (Saerens et al., 2002) |
| LIDC | Trains the size model only | Physical size is comparable between datasets, but not the rating due to context change. In the classifier LIDC made results worse |
| Validation | Grouped by CT/patient, stratified, repeated; intervals by resampling CTs; confirmation on new seeds and on the official folds | Leakage control (Kapoor and Narayanan, 2023), small sample error bars (Varoquaux, 2018), selection bias (Cawley and Talbot, 2010) |
| Metrics | The four competition metrics plus a composite (BA + F1 + QWK - MAE), within one class, sensitivity for ratings 4 and 5 | The ranking is a mean rank; the extra metrics are for our decisions and the report |

<div style="font-family: Arial, sans-serif; line-height: 1.6; color: #fab956; max-width: 1200px; margin: 0 auto; padding: 20px; border: 1px solid #000000; border-radius: 8px; background-color: #00183a74;">
  <h1 style="color: #c25719; border-bottom: 2px solid #c25719; padding-bottom: 8px; margin-top: 0;">
    References
  </h1>
  <ul style="padding-left: 20px; margin-bottom: 20px;">
    <li style="margin-bottom: 8px;">Ambroise, C. and McLachlan, G. J. (2002). Selection bias in gene extraction on the basis of microarray gene expression data. <i>PNAS</i>, 99(10), 6562 to 6566.</li>
    <li style="margin-bottom: 8px;">Armato, S. G. III et al. (2011). The Lung Image Database Consortium (LIDC) and Image Database Resource Initiative (IDRI). <i>Medical Physics</i>, 38(2), 915 to 931.</li>
    <li style="margin-bottom: 8px;">Baltatzis, V. et al. (2021). The pitfalls of sample selection: a case study on lung nodule classification. arXiv:2108.05386.</li>
    <li style="margin-bottom: 8px;">Blum, A. and Hardt, M. (2015). The Ladder: a reliable leaderboard for machine learning competitions. <i>ICML</i>.</li>
    <li style="margin-bottom: 8px;">Cawley, G. C. and Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. <i>JMLR</i>, 11, 2079 to 2107.</li>
    <li style="margin-bottom: 8px;">Collins, G. S. et al. (2024). TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <i>BMJ</i>, 385, e078378.</li>
    <li style="margin-bottom: 8px;">Danielsson, P.-E. (1980). Euclidean distance mapping. Computer Graphics and Image Processing, 14(3), 227 to 248.</li>
    <li style="margin-bottom: 8px;">Dice, L. R. (1945). Measures of the amount of ecologic association between species. <i>Ecology</i>, 26(3), 297 to 302.</li>
    <li style="margin-bottom: 8px;">Dou, Q., Chen, H., Yu, L., Qin, J. and Heng, P.-A. (2017). Multilevel contextual 3-D CNNs for false positive reduction in pulmonary nodule detection. <i>IEEE Transactions on Biomedical Engineering</i>, 64(7), 1558 to 1567.</li>
    <li style="margin-bottom: 8px;">Dwork, C. et al. (2015). The reusable holdout: preserving validity in adaptive data analysis. <i>Science</i>, 349(6248), 636 to 638.</li>
    <li style="margin-bottom: 8px;">Field, C. A. and Welsh, A. H. (2007). Bootstrapping clustered data. <i>Journal of the Royal Statistical Society B</i>, 69(3), 369 to 390.</li>
    <li style="margin-bottom: 8px;">Gneiting, T. (2011). Making and evaluating point forecasts. <i>Journal of the American Statistical Association</i>, 106(494), 746 to 762.</li>
    <li style="margin-bottom: 8px;">Gonzalez, R. C. and Woods, R. E. (2018). Digital Image Processing (4th ed.). Pearson.</li>
    <li style="margin-bottom: 8px;">Hansell, D. M. et al. (2008). Fleischner Society: glossary of terms for thoracic imaging. <i>Radiology</i>, 246(3), 697 to 722.</li>
    <li style="margin-bottom: 8px;">Johnson, W. E., Li, C. and Rabinovic, A. (2007). Adjusting batch effects in microarray expression data using empirical Bayes methods. <i>Biostatistics</i>, 8(1), 118 to 127.</li>
    <li style="margin-bottom: 8px;">Koo, T. K. and Li, M. Y. (2016). A guideline of selecting and reporting intraclass correlation coefficients for reliability research. <i>Journal of Chiropractic Medicine</i>, 15(2), 155 to 163.</li>
    <li style="margin-bottom: 8px;">Kostis, W. J., Reeves, A. P., Yankelevitz, D. F. and Henschke, C. I. (2003). Three-dimensional segmentation and growth-rate estimation of small pulmonary nodules in helical CT images. <i>IEEE Transactions on Medical Imaging</i>, 22(10), 1259 to 1274.</li>
    <li style="margin-bottom: 8px;">Kuhnigk, J.-M. et al. (2006). Morphological segmentation and partial volume analysis for volumetry of solid pulmonary lesions in thoracic CT scans. <i>IEEE Transactions on Medical Imaging</i>, 25(4), 417 to 434.</li>
    <li style="margin-bottom: 8px;">Lambin, P. et al. (2017). Radiomics: the bridge between medical imaging and personalized medicine. <i>Nature Reviews Clinical Oncology</i>, 14, 749 to 762.</li>
    <li style="margin-bottom: 8px;">MacMahon, H. et al. (2017). Guidelines for management of incidental pulmonary nodules detected on CT images: from the Fleischner Society 2017. <i>Radiology</i>, 284(1), 228 to 243.</li>
    <li style="margin-bottom: 8px;">McCullagh, P. (1980). Regression models for ordinal data. <i>Journal of the Royal Statistical Society B</i>, 42(2), 109 to 142.</li>
    <li style="margin-bottom: 8px;">Molnar, C. (2022). Interpretable Machine Learning (2nd ed.). christophm.github.io/interpretable-ml-book.</li>
    <li style="margin-bottom: 8px;">Mongan, J., Moy, L. and Kahn, C. E. (2020). Checklist for Artificial Intelligence in Medical Imaging (CLAIM). <i>Radiology: Artificial Intelligence</i>, 2(2), e200029.</li>
    <li style="margin-bottom: 8px;">Orlhac, F. et al. (2022). A guide to ComBat harmonization of imaging biomarkers in multicenter studies. <i>Journal of Nuclear Medicine</i>, 63(2), 172 to 179.</li>
    <li style="margin-bottom: 8px;">Pai, S. et al. (2024). Foundation model for cancer imaging biomarkers. <i>Nature Machine Intelligence</i>.</li>
    <li style="margin-bottom: 8px;">Pedrosa, J. et al. (2019). LNDb: a lung nodule database on computed tomography. arXiv:1911.08434.</li>
    <li style="margin-bottom: 8px;">PyRadiomics documentation, Customization (resampling and interpolators).</li>
    <li style="margin-bottom: 8px;">Regulation (EU) 2016/679 (GDPR); Regulation (EU) 2017/745 (MDR); Regulation (EU) 2024/1689 (AI Act).</li>
    <li style="margin-bottom: 8px;">Saerens, M., Latinne, P. and Decaestecker, C. (2002). Adjusting the outputs of a classifier to new a priori probabilities: a simple procedure. <i>Neural Computation</i>, 14(1), 21 to 41.</li>
    <li style="margin-bottom: 8px;">Setio, A. A. A. et al. (2016). Pulmonary nodule detection in CT images: false positive reduction using multi-view convolutional networks. <i>IEEE Transactions on Medical Imaging</i>, 35(5), 1160 to 1169.</li>
    <li style="margin-bottom: 8px;">Setio, A. A. A. et al. (2017). Validation, comparison, and combination of algorithms for automatic detection of pulmonary nodules in computed tomography images: the LUNA16 challenge. <i>Medical Image Analysis</i>.</li>
    <li style="margin-bottom: 8px;">Shafiq-ul-Hassan, M. et al. (2017). Intrinsic dependencies of CT radiomic features on voxel size and number of gray levels. <i>Medical Physics</i>, 44(3), 1050 to 1062.</li>
    <li style="margin-bottom: 8px;">Shrout, P. E. and Fleiss, J. L. (1979). Intraclass correlations: uses in assessing rater reliability. <i>Psychological Bulletin</i>, 86(2), 420 to 428.</li>
    <li style="margin-bottom: 8px;">Tejani, A. S. et al. (2024). Checklist for Artificial Intelligence in Medical Imaging (CLAIM): 2024 update. <i>Radiology: Artificial Intelligence</i>.</li>
    <li style="margin-bottom: 8px;">Traverso, A. et al. (2018). Repeatability and reproducibility of radiomic features: a systematic review. <i>International Journal of Radiation Oncology, Biology, Physics</i>, 102(4), 1143 to 1158.</li>
    <li style="margin-bottom: 8px;">van Griethuysen, J. J. M. et al. (2017). Computational radiomics system to decode the radiographic phenotype. <i>Cancer Research</i>, 77(21), e104 to e107.</li>
    <li style="margin-bottom: 8px;">Vapnik, V. and Vashist, A. (2009). A new learning paradigm: learning using privileged information. <i>Neural Networks</i>, 22(5 to 6), 544 to 557.</li>
    <li style="margin-bottom: 8px;">Varoquaux, G. (2018). Cross-validation failure: small sample sizes lead to large error bars. <i>NeuroImage</i>, 180, 68 to 77.</li>
    <li style="margin-bottom: 8px;">Wolpert, D. H. (1992). Stacked generalization. <i>Neural Networks</i>, 5(2), 241 to 259.</li>
    <li style="margin-bottom: 8px;">Zhang, Y., Jenkins, D. F., Manimaran, S. and Johnson, W. E. (2018). Alternative empirical Bayes models for adjusting for batch effects in genomic studies. <i>BMC Bioinformatics</i>, 19, 262.</li>
    <li style="margin-bottom: 8px;">Zwanenburg, A. et al. (2019). Assessing robustness of radiomic features by image perturbation. <i>Scientific Reports</i>, 9, 614.</li>
    <li style="margin-bottom: 8px;">Zwanenburg, A. et al. (2020). The Image Biomarker Standardization Initiative: standardized quantitative radiomics for high-throughput image-based phenotyping. <i>Radiology</i>, 295(2), 328 to 338.</li>
  </ul>
</div>